# Scene and scene parts

A scene is a collection of one or more scene parts. These are 3D models that can be loaded as triangle meshes or voxels.

In [ ]:
import helios

## Loading scene parts

Scene parts can be loaded from different file formats.

### 1) OBJ files

For OBJ files, you only need to provide the path to the file. If material files are referenced in the OBJ files, the material properties will also be loaded. Alternatively, users can use the `Material` class to specify and modify material properties.
Since OBJ files can have different orientations, i.e. different definitions of the "up axis", you can specify the up axis when loading the file. By default, `helios` assumes that the up axis is "z"

In [ ]:
groundplane = helios.ScenePart.from_obj("../data/sceneparts/basic/groundplane/groundplane.obj")
tree = helios.ScenePart.from_obj("../data/sceneparts/arbaro/sassafras_low.obj", up_axis="Y")  # Y as up-axis
type(tree)


helios.scene.ScenePart

### 2) TIFF files

Digital elevation models (DEMs) can be loaded as GeoTIFF files. HELIOS will use their encoded georeferencing information and resolution and build a triangle mesh. The centers of the pixels are used as data points. If all points are valid (following the *invalid*/*NoData* value definition from the GeoTIFF header), two triangles are built up: Current point - right neighbor - upper right neighbor and current point - upper neighbor - upper right neighbor:

![TIFF to mesh conversion](img/tiff_loading.png)

If any of the three points has an invalid value, the triangle is omitted.

In [ ]:
dem = helios.ScenePart.from_tiff("../data/sceneparts/tiff/dem_hd.tif")

### 3) VOX files

`helios` can read voxel models provided in a text format with .vox extension, inspired by the format used in the software [AMAPVox](https://amap-dev.cirad.fr/projects/amapvox) software (Vincent et al. 2017).
The primary purpose of this loader is to model vegetation with given leaf properties.

The content of a .vox file looks like this: 

```
    VOXEL SPACE
    #min_corner: 12.464750289916992 -10.332499504089355 243.5574951171875
    #max_corner: 21.312000274658203 -1.6010000705718994 272.84124755859375
    #split: 36 35 118
    #res:0.25 #nsubvoxel:8 #nrecordmax:0 #fraction-digits:7 #lad_type:Spherical #type:TLS #max_pad:5.0 #build-version:1.4.3
    i j k PadBVTotal angleMean bsEntering bsIntercepted bsPotential ground_distance lMeanTotal lgTotal nbEchos nbSampling transmittance attenuation attenuationBiasCorrection
    0 0 0 0 86.3137164 0.4989009 0 2.8475497 0.1240837 0.1632028 693.775004 0 4251 1 0 0
    0 0 1 0 86.989336 1.011544 0 2.840831 0.3722511 0.1784628 1292.784752 0 7244 1 0 0
    0 0 2 0 87.2224845 1.1786434 0 2.835464 0.6204185 0.1828753 1434.1079874 0 7842 1 0 0
    0 0 3 0 87.1534196 0.9608315 0 2.8170681 0.8685859 0.1816032 1186.2322972 0 6532 1 0 0
    0 0 4 0 87.2968587 0.8746295 0 2.8187793 1.1167532 0.1841085 1201.8604542 0 6528 1 0 0
    0 0 5 0 87.1479655 0.9425028 0 2.8363264 1.3649206 0.1747828 1195.6891804 0 6841 1 0 0
```

The file (as of version 1.4.3) uses space as separator and has six header lines.
- Lines 2 and 3 define the axis aligned bounding box with the xyz coordinates of the minimum and maximum corners (`#min_corner` and `#max_corner`).
- Line 4 gives the number of voxels in x, y and z direction (`#split`).
- Line 5 defines the resolution (`#res`), the leaf angle distribution type (`#lad_type`) and the maximum plant area density value (`#max_pad`). It can furthermore contain additional information which are not read by HELIOS++.
- Line 6 contains the column names. The following are relevant for HELIOS++:
    - `i`, `j`, `k`: The voxel indices in x, y and z direction.
    - `PADBVTotal` is the plant area density (m2/m3). This parameter is used by HELIOS++ in the transmittive mode to calculate the return and in the scaled mode to determine the size of each voxel (see next sections)

More in-depth explanations are given in the [AMAPVox 1.0.1 user guide](https://amap-dev.cirad.fr/attachments/download/1499/AMAPVox-1.0.1_userguide.pdf) (page 28) and in the AMAPVox GUI tooltips.

All following lines contain the voxel values with each line representing one voxel. Empty voxels (`PADBVTotal` = 0 and `transmittance` = 1) may or may not be explicitly defined.

There are three modes available for handling ray intersections with DetailedVoxels.

#### Transmittive mode (*default*)

In [ ]:
transmittive_canopy = helios.ScenePart.from_vox("../data/sceneparts/syssifoss/F_BR08_08_merged.vox", intersection_mode="transmittive")
transmittive_canopy = helios.ScenePart.from_vox("../data/sceneparts/syssifoss/F_BR08_08_merged.vox")  # since it is the default
# example with ladlut specified? 

#### Scaled mode

In [ ]:
transmittive_canopy = helios.ScenePart.from_vox("../data/sceneparts/syssifoss/F_BR08_08_merged.vox", intersection_mode="scaled", intersection_argument=0.3334)
transmittive_canopy = helios.ScenePart.from_vox("../data/sceneparts/syssifoss/F_BR08_08_merged.vox", intersection_mode="scaled", intersection_argument=0.3334, random_shift=True)

#### Fixed mode

In [ ]:
transmittive_canopy = helios.ScenePart.from_vox("../data/sceneparts/syssifoss/F_BR08_08_merged.vox", intersection_mode="fixed")

### 4) XYZ files

In [ ]:
sphere = helios.ScenePart.from_xyz("../data/sceneparts/pointclouds/sphere_dens25000.xyz", voxel_size=0.1)

### Loading multiple scene parts at one

Of course, these methods can be put into a loop to load a large number of scene parts at once. However, `helios` also supports loading multiple scene parts by specifying a path with wildcards: 

In [ ]:
toyblocks = helios.ScenePart.from_objs("../data/sceneparts/toyblocks/*.obj")
type(toyblocks)

In [ ]:
# TODO: continue here